# Introduction

Dependencies

In [ ]:
import pandas as pd
import numpy as np

A 25-column dataset on customer-leads is provided

## Preprocessing

We prescribe some ideas as to how to handle different variables in our analysis process. We also recode a few (mostly into integers) for ease of analysis, interpretation and communication.

In [ ]:
rawdf = pd.read_csv('Analyst_case_study_dataset_1.csv')

### 1.  `LeadCreated`

Kept intact.

In [ ]:
df = rawdf

df['Date'] = pd.to_datetime(df['LeadCreated'])
df.drop(columns='LeadCreated', inplace=True)

### 2.  `FirstName`

Dropped. (Note: Labelling generic or spammy names to predict lead quality could be a useful future enhancement.)

### 3.  `Email`

Kept intact. Interactions can be grouped by email, assuming an entity consistently uses the same email.

In [ ]:
email = df['Email'].str.strip()

df['EmailUname'] = email.str.split('@', n=1).str[0]

domain = email.str.split('@', n=1).str[1]

df['EmailSite'] = domain.str.rsplit('.', n=1).str[0]
df['EmailDom'] = domain.str.rsplit('.', n=1).str[1]

df.drop(columns='Email', inplace=True)

### 4.  `VendorLeadID`

Kept intact to group multiple interaction events together.

### 5.  `CallStatus`

Recoded into a new variable called `CallStatusNo` as follows (where higher scores indicate greater "closeness" to a closed lead):
- `NaN`: Kept intact.
- `Unable to contact - Bad Contact Information`: 1 (Lowest rung of quality; unusable lead).
- `Contacted - Invalid Profile`: 2 (Nearly hopeless, unless prospective customers decline with false excuses).
- `Contacted - Doesn't Qualify`: 3 (Some potential for future interest).
- `EP Sent`: 4
- `EP Received`: 5
- `EP Confirmed`: 6
- `Closed`: 7 (Highest level of success).

In [ ]:
df.CallStatus.unique()

<ArrowStringArray>
[                                          nan,
 'Unable to contact - Bad Contact Information',
                 'Contacted - Doesn't Qualify',
                                      'Closed',
                                 'EP Received',
                                'EP Confirmed',
                 'Contacted - Invalid Profile',
                                     'EP Sent']
Length: 8, dtype: str

In [ ]:
df['CallStatusNo'] = df['CallStatus'].map({
    # np.nan:0,
    'Unable to contact - Bad Contact Information':1,
    'Contacted - Invalid Profile':2,
    'Contacted - Doesn\'t Qualify':3,
    'EP Sent':4,
    'EP Received':5,
    'EP Confirmed':6,
    'Closed':7
})

df.drop(columns='CallStatus', inplace=True)

### 6.  `WidgetName`

First 3 common segments removed. The next 3 segments are split and expanded into new columns: `FormPgsCnt`, `Fieldset`, and `ColorDesign`.

In [ ]:
widget_list = df['WidgetName'].str.split('-', n=4, expand=True).drop(columns = [0,1,2])
# widget_list[3].unique()
widget_list.columns = ['FormSingle','Fieldset']
df = df.join(widget_list)
df['FormSingle'] = df['FormSingle'].str.replace('DC', '').astype(int).replace(2,0)

df.drop(columns='WidgetName', inplace=True)

### 7.  `PublisherZoneName`

First segment (common display-size info) removed. The second segment (position) is recoded into a binary variable (0/1) since it only has 2 alternatives.

In [ ]:
zone_list = df['PublisherZoneName'].str.split('-', n=1, expand=True)[0]

zone_list = (zone_list=='TopLeft').astype(int)
zone_list
df['TLZone'] = zone_list

df.drop(columns='PublisherZoneName', inplace=True)

### 8.  `PublisherCampaignName`

Recoded into a binary variable (2 alternatives).

In [ ]:
df['PubCampaign'] = (df['PublisherCampaignName'] == 'DebtReductionInc').astype(int)

df.drop(columns='PublisherCampaignName', inplace=True)

### 9.  `AddressScore`

Missing values imputed via linear regression using other variables. Serves as a predictor for `CallStatusNo`, as higher scores reflect consumer sincerity.

### 10. `PhoneScore`

Missing values imputed via linear regression using other variables. Serves as a predictor for `CallStatusNo`, as higher scores reflect consumer sincerity.

In [ ]:
# df['PhoneScore'] = 

# df['AddressScore'] = 

### 11. `AdvertiserCampaignName`

Recoded into a binary variable.

In [ ]:
df['AdCampaign'] = (df['AdvertiserCampaignName'] == 'Debt Settlement1 Master').astype(int)

df.drop(columns='AdvertiserCampaignName', inplace=True)

### 12. `State`

Kept intact. Can be 0-1 encoded when necessary, for NN models, say.

### 13. `Debt level`

Converted into an ordinal variable ranging from 1 to 7. (An alternative approach using class means can be considered, carefully handling the right open-ended final class)


In [ ]:
df['DebtLevelNo'] = df['DebtLevel'].map({
    '7500-10000':0.08,
    '7500-15000':0.11,
    '10001-15000':0.125,
    '15001-20000':0.17,
    '20001-30000':0.25,
    '30001-50000':0.4,
    '50001-70000':0.6,
    '70001-90000':0.8,
    '90000-100000':0.95,
    'More_than_100000':1,
})

df.drop(columns='DebtLevel', inplace=True)

### 14. `IP Address`

Dropped. (Filtering spammy or malevolent IPs can be tested in advanced stages).

In [ ]:
df.drop(columns=['IP Address', ], inplace=True)

### 15. `Partner`

Kept intact.

In [ ]:
# df['Partner'].unique()
df['Partner'].str.replace('google', 'Google')

0            Google
1       AdKnowledge
2       AdKnowledge
3            Google
4            Google
           ...     
3016          yahoo
3017         Google
3018          yahoo
3019         Google
3020         Google
Name: Partner, Length: 3021, dtype: str

### 16. `MarketingCampaign`

In [ ]:
# df['MarketingCampaign'].unique().tolist()
campaign_groups = {
    'DHT': [
        'Debt Holding Tank',
        'Debt_Holding_Tank_-_SN'
    ],

    'FS': [
        'Financial Services'
    ],

    'DRI': [
        'DebtReductionInc',
        'DebtReductionInc - YSM Restructure (05/29/09)'
    ],

    'C': [
        'Credit',
        'Credit_-_SN'
    ],

    'DV': [
        'Debt Volume',
        'Debt Volume_MI',
        'Debt Volume_TX',
        'Debt Volume_CA',
        'Debt_Volume_-_SN'
    ],

    'DG': [
        'Debt General',
        'Debt_General_-_SN'
    ],

    'S': [
        'state',
        'state - YSM Restructure (05/29/09)'
    ],

    'DC': [
        'Debt Consolidation',
        'Debt_Consolidation_-_SN'
    ],

    'D': [
        'Debt'
    ],

    'DR': [
        'Debt Reduction'
    ]
}

campaign_map = {
    variant: main
    for main, variants in campaign_groups.items()
    for variant in variants
}

df['MarketingCampaign'] = df['MarketingCampaign'].replace(campaign_map)

### 18. `AdGroup`

In [ ]:
df['AdGroupVol'] = 0

df.loc[
    df['AdGroup'].str.contains(r'\b(HV|high volume)\b', case=False, na=False),
    'AdGroupVol'
] = 1

df.loc[
    df['AdGroup'].str.contains(r'\bLV\b', case=False, na=False),
    'AdGroupVol'
] = -1

df.drop(columns='AdGroup', inplace=True)

/tmp/ipykernel_20649/2805233099.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['AdGroup'].str.contains(r'\b(HV|high volume)\b', case=False, na=False),


### 18. Rest of the columns

(`ReferralDomain`, `MarketingCampaign`, `AdGroup`, `Keyword`, `SearchQuery`, `Referral URL`, `ReferralURL Parameters`, `LandingPageURL`, `LandingPageURL parameters`): Dropped for the time being.

181 days = 20 weeks. Weekly aggregation is meaningful when necessary.

## Keyword and rest

In [23]:
# df['Keyword'].unique().tolist()    

df.drop(columns='Keyword', inplace=True)

In [24]:
unhandled = ['SearchQuery', 'ReferralURL', 'ReferralURL', 'ReferralURL Parameters', 'LandingPageURL', 'Landing Page URL Parameters']

df.drop(columns = ['FirstName']+unhandled, inplace=True)

In [25]:
df.to_parquet('processed/df1_preproc.parquet', index=False)
df.to_csv('processed/df1_preproc.csv', index=False)

# Imputation

In [26]:
nonnum_cols = ['Email', 'VendorLeadID', 'Date', 'CallStatusNo']

In [27]:
df[['PhoneScore', 'AddressScore']] = df[['PhoneScore', 'AddressScore']].fillna(3)

In [28]:
df.to_parquet('processed/df2_imputed.parquet', index=False)

In [29]:
# from sklearn.compose import ColumnTransformer
# from sklearn.preprocessing import OneHotEncoder

# # Define all categorical columns you want to encode
# cat_cols = ['Partner', 'MarketingCampaign', 'Fieldset']
# #cat_cols = ['Partner', 'MarketingCampaign', 'Fieldset', 'State']  # Just append new column names here as you grow!

# # Create the transformer
# preprocessor = ColumnTransformer(
#     transformers=[
#         ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_cols)
#     ],
#     remainder='passthrough',
#     verbose_feature_names_out=False,
# ).set_output(transform='pandas')

# # Transform the dfframe in one shot
# df = preprocessor.fit_transform(df)